In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# LABORATOR U10 - SERIE LUNARA: MEDII MOBILE, RATE, INDICI, VALORI REALE
# ============================================================

# PARAMETRI (Cartea de identitate)
PARAMETRI_INDICATOR = {
    'variabila':          'Încasările lunare ale școlii private (mii lei)',
    'unitate':            'mii lei (valori nominale)',
    'periodicitate':      'lunar',
    'perioada':           'ianuarie 2023 - decembrie 2025',
    'an_baza':            2023,          
    'deflator':           'IPC (Indicele Prețurilor de Consum)',
    'metoda':             'medie mobila 12 luni, rate, indici baza fixa',
    'rata_comunicata':    'rata anuala (%)',
    'lungime_medie_mobila': 12,          
    'verificari_fisa': {
        'iunie 2024':      {'medie_mobila': 66.58, 'rata_lunara': -3.70, 'rata_anuala': 7.84},
        'iunie 2025':      {'medie_mobila': 74.00, 'rata_lunara': 8.62, 'rata_anuala': 5.45},
        'decembrie 2025':  {'medie_mobila': 82.67, 'rata_lunara': -12.50, 'rata_anuala': 7.14},
    },
}

# Helper pentru parsarea lunilor din română
LUNI_RO = {
    'ianuarie': 1, 'februarie': 2, 'martie': 3, 'aprilie': 4,
    'mai': 5, 'iunie': 6, 'iulie': 7, 'august': 8,
    'septembrie': 9, 'octombrie': 10, 'noiembrie': 11, 'decembrie': 12,
}

def eticheta_la_period(eticheta: str) -> pd.Period:
    parti = eticheta.lower().strip().split()
    luna_nr = LUNI_RO[parti[0]]
    an = int(parti[1])
    return pd.Period(f"{an}-{luna_nr:02d}", freq='M')

# DATE REALE DIN FOTOGRAFIE (Tabelul)
luni_str = pd.period_range('2023-01', '2025-12', freq='M').astype(str)
valori = [
    64, 59, 52, 51, 55, 51, 59, 81, 115, 107, 75, 91,  # 2023
    70, 63, 53, 54, 60, 55, 64, 88, 124, 110, 81, 98,  # 2024
    73, 65, 58, 59, 63, 58, 68, 93, 134, 121, 85, 105  # 2025
]
ipc_valori = [
    97.8, 98.2, 98.6, 99.0, 99.4, 99.7, 100.2, 100.6, 101.0, 101.4, 101.8, 102.3, # 2023
    102.7, 103.1, 103.5, 103.9, 104.4, 104.8, 105.2, 105.6, 106.1, 106.5, 106.9, 107.4, # 2024
    107.8, 108.3, 108.7, 109.1, 109.6, 110.0, 110.5, 110.9, 111.4, 111.8, 112.3, 112.7  # 2025
]

cadru_date_u10 = pd.DataFrame({
    'luna': luni_str,
    'an': pd.PeriodIndex(luni_str, freq='M').year,
    'valoare': valori,
    'indice_preturi': ipc_valori
})

# PASUL 0 - Pregatirea datelor
cadru_date_u10['luna'] = pd.PeriodIndex(cadru_date_u10['luna'], freq='M')
cadru_date_u10 = cadru_date_u10.sort_values('luna').reset_index(drop=True)

print("PAS 0 - Date incarcate")
print(cadru_date_u10.head(5).to_string(index=False))
print(f"Total luni: {len(cadru_date_u10)}")

# PASUL 1 - Media anuala si extreme
media_anuala = cadru_date_u10.groupby('an')['valoare'].mean().round(2)
extrem_max = (cadru_date_u10.loc[cadru_date_u10.groupby('an')['valoare'].idxmax(), ['an', 'luna', 'valoare']]
              .rename(columns={'luna': 'luna_max', 'valoare': 'val_max'}))
extrem_min = (cadru_date_u10.loc[cadru_date_u10.groupby('an')['valoare'].idxmin(), ['an', 'luna', 'valoare']]
              .rename(columns={'luna': 'luna_min', 'valoare': 'val_min'}))
tabel_anual = (extrem_max.merge(extrem_min, on='an').assign(media_anuala=media_anuala.values))

print("\nPASUL 1 - Media anuala, luna max/min")
print(tabel_anual.to_string(index=False))

# PASUL 2 - Media mobila pe 12 luni
n_mm = PARAMETRI_INDICATOR['lungime_medie_mobila']
cadru_date_u10['medie_mobila'] = cadru_date_u10['valoare'].rolling(window=n_mm, min_periods=n_mm).mean().round(2)

print("\nPASUL 2 - Media mobila 12 luni")
print(cadru_date_u10.loc[cadru_date_u10['medie_mobila'].notna(), ['luna', 'valoare', 'medie_mobila']].tail(3).to_string(index=False))

# Grafic
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cadru_date_u10['luna'].astype(str), cadru_date_u10['valoare'], marker='o', label='Serie')
ax.plot(cadru_date_u10['luna'].astype(str), cadru_date_u10['medie_mobila'], label='MM 12 luni')
ax.set_title('Incasari lunare si medie mobila'); ax.legend(); ax.tick_params(axis='x', rotation=90)
plt.tight_layout(); plt.savefig('grafic_u10.png', dpi=100); plt.close()
print("Grafic salvat: grafic_u10.png")

# PASUL 3 - Rate lunare si anuale
cadru_date_u10['rata_lunara'] = cadru_date_u10['valoare'].pct_change() * 100
cadru_date_u10['rata_anuala'] = cadru_date_u10['valoare'].pct_change(n_mm) * 100

dif_rate = (cadru_date_u10['rata_anuala'] - cadru_date_u10['rata_lunara']).abs()
idx_max = dif_rate.idxmax()

print("\nPASUL 3 - Rate lunare si anuale")
print(f"Luna cu cea mai mare diferenta: {cadru_date_u10.loc[idx_max, 'luna']} ({dif_rate.iloc[idx_max]:.2f} pp)")
for eticheta in ['iunie 2024', 'iunie 2025', 'decembrie 2025']:
    luna_cautata = eticheta_la_period(eticheta)
    rand = cadru_date_u10.loc[cadru_date_u10['luna'] == luna_cautata]
    if not rand.empty:
        print(f"  {eticheta}: rata_lunara={rand['rata_lunara'].iloc[0]:.2f}% | rata_anuala={rand['rata_anuala'].iloc[0]:.2f}%")

# PASUL 4 - Indicii (baza fixa si in lant)
an_baza = PARAMETRI_INDICATOR['an_baza']
media_baza = cadru_date_u10.loc[cadru_date_u10['an'] == an_baza, 'valoare'].mean()
cadru_date_u10['indice_baza_fixa'] = (cadru_date_u10['valoare'] / media_baza * 100).round(2)
cadru_date_u10['indice_lant'] = (cadru_date_u10['valoare'].pct_change() + 1).mul(100).round(2)

print("\nPASUL 4 - Indicii")
print(f"Media anului de baza {an_baza}: {media_baza:.2f}")
print(cadru_date_u10[['luna', 'valoare', 'indice_baza_fixa', 'indice_lant']].tail(3).to_string(index=False))

# PASUL 5 - Valori reale si descompunere
cadru_date_u10['valoare_reala'] = (cadru_date_u10['valoare'] / cadru_date_u10['indice_preturi'] * 100).round(2)
cadru_date_u10['rata_anuala_reala'] = cadru_date_u10['valoare_reala'].pct_change(n_mm) * 100
cadru_date_u10['efect_preturi'] = cadru_date_u10['indice_preturi'].pct_change(n_mm) * 100
cadru_date_u10['efect_volum'] = cadru_date_u10['rata_anuala_reala']

print("\nPASUL 5 - Valori reale")
print(cadru_date_u10.dropna(subset=['rata_anuala_reala']).tail(3)[['luna', 'rata_anuala', 'rata_anuala_reala', 'efect_preturi', 'efect_volum']].to_string(index=False))

# PASUL 6 - Raport (ИСПРАВЛЕНО: используется eticheta_la_period)
print("\nPASUL 6 - Raport final")
cazuri_raport = []
for eticheta in ['iunie 2024', 'iunie 2025', 'decembrie 2025']:
    luna_cautata = eticheta_la_period(eticheta)
    rand = cadru_date_u10.loc[cadru_date_u10['luna'] == luna_cautata]
    if rand.empty:
        continue
    r = rand.iloc[0]
    cazuri_raport.append(('Medie mobila 12 luni', eticheta, round(r['medie_mobila'], 2)))
    cazuri_raport.append(('Rata lunara (%)', eticheta, round(r['rata_lunara'], 2)))
    cazuri_raport.append(('Rata anuala (%)', eticheta, round(r['rata_anuala'], 2)))
    cazuri_raport.append(('Indice baza fixa', eticheta, round(r['indice_baza_fixa'], 2)))
    cazuri_raport.append(('Rata anuala reala (%)', eticheta, round(r['rata_anuala_reala'], 2) if pd.notna(r['rata_anuala_reala']) else None))

tabel_raport_u10 = pd.DataFrame(cazuri_raport, columns=['Indicator', 'Luna', 'Valoare'])

concluzie = ("Concluzie: Incasarile cresc nominal, dar inflatia reduce cresterea reala.")
tabel_raport_u10.loc[len(tabel_raport_u10)] = ['Concluzie', '', concluzie]

print(tabel_raport_u10.to_string(index=False))
tabel_raport_u10.to_csv('raport_u10.csv', index=False)
print("\nRaport salvat in: raport_u10.csv")

PAS 0 — Date încărcate
   luna   an  luna_nr  valoare  indice_preturi
2023-01 2023        1    102.5            99.6
2023-02 2023        2    102.7           100.8
2023-03 2023        3    106.1           101.2
2023-04 2023        4    108.2           100.9
2023-05 2023        5    106.6           101.6
Total luni: 36

PASUL 1 — Media anuală, luna max/min
  an luna_max  val_max luna_min  val_min  media_anuala
2023  2023-04    108.2  2023-08    100.9        104.72
2024  2024-12    120.3  2024-01    111.0        115.18
2025  2025-05    129.9  2025-01    120.5        124.40

PASUL 2 — Media mobilă 12 luni
Ferestre verificate: 24 | toate corecte: True
   luna  valoare  medie_mobila
2025-08    125.7        121.88
2025-09    120.6        122.32
2025-10    124.1        123.23
2025-11    121.7        123.88
2025-12    126.5        124.40
Grafic salvat: grafic_u10.png

PASUL 3 — Rate lunare și anuale
Luna cu cea mai mare diferență |rata_anuala − rata_lunara|: 2024-08 (14.22 pp)

PASUL 4 — Indic

DateParseError: Unknown datetime string format, unable to parse: IUNIE-2024